# MarkupLM phishing fine-tuning on the Laya 5k split

Fine-tunes `microsoft/markuplm-base` (`MarkupLMForSequenceClassification`, one label per page)
on **the same PhreshPhish split Laya was trained on**, so the two models can be compared
fairly. MarkupLM `v1` only ever saw 800 pages; this run gives it the full training set.

What is held equal with `finetune_laya_phishing_kaggle_v2.ipynb`:

| | Laya | This notebook |
|---|---|---|
| Sampling code, seed, shard counts | section 2 | copied verbatim |
| Train / val / test pages | 5,000 / 1,000 / 1,000 per class | same pages |
| HTML cleaning | scripts, styles, noscript, templates, comments stripped; 300k chars | same |
| Checkpoint selection | lowest validation loss, early stopping 2 | same |
| Effective batch | 16 | 16 |

What differs, because the models differ: MarkupLM reads text nodes plus their XPaths rather
than raw markup, and uses a 2,048-token window (MarkupLM `v1`'s recipe) against Laya's 1,024.

Two comparisons come out of this:

1. **PhreshPhish test AUC** (section 8). If you attach the Laya artifact, the notebook checks
   that both models were tested on identical pages before printing the comparison.
2. **The 300 Mendeley pages** — the fair, never-seen corpus. Run `smoke_test.ipynb` on the
   saved artifact afterwards; Laya v3 scored 0.798 there and MarkupLM `v1` 0.814.

Run on a Kaggle GPU (T4 or better): Settings → Accelerator → GPU. Run all cells in order.

In [ ]:
%pip install -q "transformers>=4.48.0" pyarrow scikit-learn beautifulsoup4 huggingface_hub

In [ ]:
import torch, transformers
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
print("transformers", transformers.__version__)
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - enable a GPU accelerator")
# markuplm-base ships .bin weights only; transformers refuses to torch.load them below 2.6.
assert tuple(int(p) for p in torch.__version__.split("+")[0].split(".")[:2]) >= (2, 6), "PyTorch >= 2.6 required"

## 1. Configuration

Hyperparameters are MarkupLM `v1`'s (`train_html_phishing_classifier.ipynb`) except the
batch: micro-batches of 2 accumulated 8 times give the same effective batch of 16 as Laya.
If you hit an out-of-memory error, set `BATCH_SIZE = 1` and `GRAD_ACCUM_STEPS = 16`.

`LAYA_ARTIFACT_DIR` is optional. Point it at the Laya artifact folder (attached as a Kaggle
input) to verify the test split and compare AUCs side by side. It needs that folder's
`metadata.json` and `split_manifest.jsonl`.

In [ ]:
import json, math, random, hashlib, re, os, time
from pathlib import Path
from collections import Counter, defaultdict
from contextlib import nullcontext
from urllib.parse import urlsplit

import numpy as np
import pyarrow.parquet as pq
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from bs4 import BeautifulSoup, Comment
from huggingface_hub import HfApi, hf_hub_download
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, roc_auc_score
from transformers import (MarkupLMFeatureExtractor, MarkupLMForSequenceClassification,
                          MarkupLMProcessor, get_linear_schedule_with_warmup)
from importlib.metadata import version

SEED = 42
# The sampling cell below is copied verbatim from the Laya v2 notebook and expects this mapping.
PHRESHPHISH_LABEL_TO_ID = {"benign": 0, "phish": 1}
random.seed(SEED); torch.manual_seed(SEED); np.random.seed(SEED)

BASE_MODEL = "microsoft/markuplm-base"
ARTIFACT_DIR = Path("markuplm-phishing-5k")   # download from Kaggle output
LAYA_ARTIFACT_DIR = None                       # e.g. Path("/kaggle/input/laya-v3/laya_v3")

LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.1
MAX_GRAD_NORM = 1.0
NUM_EPOCHS = 3                 # maximum; best validation-loss weights are restored
EARLY_STOPPING_PATIENCE = 2
MAX_TOKENS = 2048              # MarkupLM v1's window; section 5 widens the 512 position table
BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 8           # effective batch 16, as Laya
CURRENT_BANDS = {"warn_from": 0.4, "block_from": 0.6}   # backend/app/risk.py today

# Cutoff targets (section 8). The product team sets these, not the model.
BLOCK_MAX_FPR = 0.02     # block at most 2% of safe validation pages
WARN_MIN_RECALL = 0.90   # warn on at least 90% of phishing validation pages

id2label = {0: "benign", 1: "phishing"}
label2id = {v: k for k, v in id2label.items()}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    raise RuntimeError("This notebook needs a GPU. On Kaggle: Settings > Accelerator > GPU.")
# T4 has no bfloat16, so fall back to float16 with a gradient scaler.
AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print("device:", torch.cuda.get_device_name(0), "| amp:", AMP_DTYPE)


def amp():
    return torch.autocast("cuda", dtype=AMP_DTYPE)

## 2. Sampling — identical to Laya v2

This cell is a verbatim copy of section 2 of `finetune_laya_phishing_kaggle_v2.ipynb`:
same seed, shard counts, per-class targets and cache directory. Sampling is deterministic,
so it reproduces Laya's train/val/test pages. Section 8 checks that against Laya's saved
manifest when one is attached; don't edit this cell, or the comparison stops being fair.

The first run downloads 24 parquet shards (several GB) and takes a while.

In [ ]:
DATASET_ID = "phreshphish/phreshphish"
CACHE_DIR = Path("datasets/phreshphish_sample/laya_v2_expanded")
USE_TRAIN_PER_CLASS = 5_000
USE_VAL_PER_CLASS = 1_000
USE_TEST_PER_CLASS = 1_000
TRAIN_SHARDS = 16  # evenly spread across available shards; None scans all
TEST_SHARDS = 8   # evenly spread across available shards; None scans all
MIN_HTML_CHARS = 20
MAX_HTML_CHARS = 1_000_000
PARSE_CHAR_BUDGET = 300_000
SHORT_HTML_CHARS = 2_000
SHORT_FRACTION = 0.20
BENIGN_LOGIN_FRACTION = 0.20
LOGIN_RE = re.compile(r"<input\b[^>]*\btype\s*=\s*['\"]?password\b", re.I)


def normalize_page(row):
    html = row.get("html") or ""
    label = row.get("label")
    if label not in PHRESHPHISH_LABEL_TO_ID or not MIN_HTML_CHARS <= len(html) <= MAX_HTML_CHARS:
        return None
    url = row.get("url") or ""
    try:
        host = (urlsplit(url).hostname or "").lower().rstrip(".")
    except ValueError:
        host = ""
    # Hash before truncation, so different tails are not mistaken for identical pages.
    digest = hashlib.sha256(html.encode("utf-8", "ignore")).hexdigest()
    return {"html": html[:PARSE_CHAR_BUDGET], "html_len": len(html),
            "label": label, "label_id": PHRESHPHISH_LABEL_TO_ID[label], "url": url,
            "host": host, "sha256": digest,
            "is_login": bool(LOGIN_RE.search(html)), "short": len(html) <= SHORT_HTML_CHARS}


def host_group(row):
    return "host:" + row["host"] if row["host"] else "hash:" + row["sha256"]


def reservoir_add(bucket, row, count, capacity, rng):
    if len(bucket) < capacity:
        bucket.append(row)
    else:
        position = rng.randrange(count)
        if position < capacity:
            bucket[position] = row


def select_coverage(uniform, short, login, limit, short_fraction, login_fraction):
    selected, seen = [], set()
    def add(rows, target):
        added = 0
        for row in rows:
            if len(selected) >= limit or added >= target:
                break
            if row["sha256"] not in seen:
                selected.append(row)
                seen.add(row["sha256"])
                added += 1
    add(short, round(limit * short_fraction))
    add(login, round(limit * login_fraction))
    add(uniform, limit)
    return selected


def sample_shard(local_path, per_class, seed, enrich):
    rng = random.Random(seed)
    pools = {(label, kind): [] for label in (0, 1) for kind in ("uniform", "short", "login")}
    counts = Counter()
    for batch in pq.ParquetFile(local_path).iter_batches(batch_size=64, columns=["html", "url", "label"]):
        for raw in batch.to_pylist():
            row = normalize_page(raw)
            if row is None:
                continue
            label = row["label_id"]
            kinds = ["uniform"]
            if enrich and row["short"]:
                kinds.append("short")
            if enrich and label == 0 and row["is_login"]:
                kinds.append("login")
            for kind in kinds:
                key = (label, kind)
                counts[key] += 1
                reservoir_add(pools[key], row, counts[key], per_class, rng)
    selected = []
    for label in (0, 1):
        for kind in ("uniform", "short", "login"):
            rng.shuffle(pools[(label, kind)])
        selected += select_coverage(
            pools[(label, "uniform")], pools[(label, "short")], pools[(label, "login")],
            per_class, SHORT_FRACTION if enrich else 0,
            BENIGN_LOGIN_FRACTION if enrich and label == 0 else 0,
        )
    return selected


def select_shard_indices(total, requested):
    if total < 1:
        raise ValueError("No shards available")
    if requested is not None and (not isinstance(requested, int) or isinstance(requested, bool) or requested < 1):
        raise ValueError("Shard count must be a positive integer or None for all")
    count = total if requested is None else min(requested, total)
    return [round(i * (total - 1) / max(count - 1, 1)) for i in range(count)]


def sample_split(split, per_class, shard_count, enrich):
    files = sorted(f for f in HfApi().list_repo_files(DATASET_ID, repo_type="dataset")
                   if f.startswith(f"data/{split}-") and f.endswith(".parquet"))
    if not files:
        raise RuntimeError(f"No parquet shards for {split}")
    indices = select_shard_indices(len(files), shard_count)
    count = len(indices)
    print(f"{split}: scanning {count}/{len(files)} shards; candidate target {per_class:,} per class")
    print("Whole selected shards are downloaded and scanned; only sampled rows enter training.")
    rows = []
    for index in indices:
        shard = files[index]
        local = hf_hub_download(DATASET_ID, shard, repo_type="dataset")
        sampled = sample_shard(local, math.ceil(per_class / count), SEED + index, enrich)
        for row in sampled:
            row["shard"] = shard
        rows.extend(sampled)
        print(shard, "selected:", len(sampled))
    return rows


def deduplicate(rows):
    by_hash, conflicting = {}, set()
    for row in rows:
        key = row["sha256"]
        if key in by_hash and by_hash[key]["label_id"] != row["label_id"]:
            conflicting.add(key)
        by_hash[key] = row
    return [r for key, r in by_hash.items() if key not in conflicting]


def choose_balanced(rows, per_class, seed, enrich=False):
    rng = random.Random(seed)
    result = []
    for label in (0, 1):
        group = [r for r in rows if r["label_id"] == label]
        rng.shuffle(group)
        result += select_coverage(group, [r for r in group if r["short"]],
                                  [r for r in group if r["is_login"] and label == 0], per_class,
                                  SHORT_FRACTION if enrich else 0,
                                  BENIGN_LOGIN_FRACTION if enrich and label == 0 else 0)
    rng.shuffle(result)
    return result


def split_hosts(rows, val_per_class, seed):
    groups = defaultdict(list)
    for row in rows:
        groups[host_group(row)].append(row)
    keys = sorted(groups)
    random.Random(seed).shuffle(keys)
    val, val_groups, counts = [], set(), Counter()
    for key in keys:
        group = groups[key]
        additions = Counter(r["label_id"] for r in group)
        if all(counts[label] + additions[label] <= val_per_class for label in (0, 1)):
            val.extend(group)
            counts.update(additions)
            val_groups.add(key)
    return [r for r in rows if host_group(r) not in val_groups], val


def assert_separate(splits):
    previous_hosts, previous_hashes = set(), set()
    for name, rows in splits.items():
        hosts, hashes = {host_group(r) for r in rows}, {r["sha256"] for r in rows}
        if hosts & previous_hosts or hashes & previous_hashes:
            raise ValueError(f"Host/content overlap in {name}")
        if len(hashes) != len(rows):
            raise ValueError(f"Duplicate content in {name}")
        if set(r["label_id"] for r in rows) != {0, 1}:
            raise ValueError(f"{name} needs both classes; increase shard/sample counts")
        previous_hosts.update(hosts)
        previous_hashes.update(hashes)


sample_config = {"version": 2, "dataset": DATASET_ID, "seed": SEED,
                 "train_shards": TRAIN_SHARDS, "test_shards": TEST_SHARDS,
                 "train_target_per_class": USE_TRAIN_PER_CLASS,
                 "val_target_per_class": USE_VAL_PER_CLASS,
                 "test_target_per_class": USE_TEST_PER_CLASS,
                 "train_pool_per_class": 2 * (USE_TRAIN_PER_CLASS + USE_VAL_PER_CLASS),
                 "test_pool_per_class": 3 * USE_TEST_PER_CLASS,
                 "min_chars": MIN_HTML_CHARS, "max_chars": MAX_HTML_CHARS,
                 "parse_budget": PARSE_CHAR_BUDGET, "short_chars": SHORT_HTML_CHARS,
                 "short_fraction": SHORT_FRACTION, "login_fraction": BENIGN_LOGIN_FRACTION}
cache_metadata = CACHE_DIR / "sampling.json"
cache_valid = (cache_metadata.exists()
               and json.loads(cache_metadata.read_text(encoding="utf-8")) == sample_config
               and all((CACHE_DIR / f"{name}.jsonl").exists() for name in ("pool", "test")))
if cache_valid:
    print("Reusing matching expanded sampling cache:", CACHE_DIR)
    def read_cache(name):
        with (CACHE_DIR / f"{name}.jsonl").open(encoding="utf-8") as handle:
            return [json.loads(line) for line in handle]
    pool, test_pool = read_cache("pool"), read_cache("test")
else:
    print("Building expanded sample; original small cache is preserved.")
    pool = sample_split("train", sample_config["train_pool_per_class"], TRAIN_SHARDS, True)
    test_pool = sample_split("test", sample_config["test_pool_per_class"], TEST_SHARDS, False)
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    for name, rows in (("pool", pool), ("test", test_pool)):
        with (CACHE_DIR / f"{name}.jsonl").open("w", encoding="utf-8") as handle:
            for row in rows:
                handle.write(json.dumps(row, ensure_ascii=False) + "\n")
    cache_metadata.write_text(json.dumps(sample_config, indent=2), encoding="utf-8")

pool = deduplicate(pool)
train_pool, val_raw = split_hosts(pool, USE_VAL_PER_CLASS, SEED)
train_raw = choose_balanced(train_pool, USE_TRAIN_PER_CLASS, SEED, enrich=True)
known_hosts = {host_group(r) for r in train_raw + val_raw}
known_hashes = {r["sha256"] for r in pool}
test_pool = deduplicate(test_pool)
eligible_test = [r for r in test_pool if host_group(r) not in known_hosts and r["sha256"] not in known_hashes]
print("Excluded test pages with seen hosts/content:", len(test_pool) - len(eligible_test))
test_raw = choose_balanced(eligible_test, USE_TEST_PER_CLASS, SEED + 1)
assert_separate({"train": train_raw, "val": val_raw, "test": test_raw})
sampling_report = {}
targets = {"train": USE_TRAIN_PER_CLASS, "val": USE_VAL_PER_CLASS, "test": USE_TEST_PER_CLASS}
for name, rows in (("train", train_raw), ("val", val_raw), ("test", test_raw)):
    counts = Counter(r["label_id"] for r in rows)
    sampling_report[name] = {
        "target_per_class": targets[name], "actual_per_class": dict(counts),
        "actual_total": len(rows), "unique_hosts": len({r["host"] for r in rows}),
        "represented_shards": sorted({r["shard"] for r in rows}),
    }
    if any(counts[label] < targets[name] for label in (0, 1)):
        print(f"WARNING: {name} below target after deduplication/host isolation: {dict(counts)}. "
              "Increase shard coverage or candidate-pool multipliers before relying on target counts.")
    print(name, "labels:", dict(Counter(r["label_id"] for r in rows)),
          "short benign:", sum(r["short"] and r["label_id"] == 0 for r in rows),
          "benign login:", sum(r["is_login"] and r["label_id"] == 0 for r in rows))

## 3. HTML preprocessing

Cleaning is Laya's `clean_html`, character for character. MarkupLM then splits the cleaned
markup into text nodes and their XPaths. A page with markup but no visible text is usable
for Laya but not for MarkupLM; such pages are counted as parse failures and reported.

In [ ]:
STRIP_TAGS = ["script", "style", "noscript", "template"]
feature_extractor = MarkupLMFeatureExtractor()


def clean_html(html):
    soup = BeautifulSoup(html[:PARSE_CHAR_BUDGET], "html.parser")
    for element in soup.find_all(STRIP_TAGS):
        element.decompose()
    for comment in soup.find_all(string=lambda text: isinstance(text, Comment)):
        comment.extract()
    text = str(soup)
    if not text.strip():
        raise ValueError("No usable markup")
    return text


def extract_page(html):
    encoding = feature_extractor(clean_html(html))
    pairs = [(node.strip(), xpath) for node, xpath in zip(encoding["nodes"][0], encoding["xpaths"][0])
             if node.strip()]
    if not pairs:
        raise ValueError("No usable text nodes")
    nodes, xpaths = zip(*pairs)
    return {"nodes": [list(nodes)], "xpaths": [list(xpaths)]}


parse_failures = []


def build_examples(raw_examples, split):
    examples = []
    for item in tqdm(raw_examples, desc=f"parse {split}"):
        try:
            parsed = extract_page(item["html"])
        except Exception as exc:
            parse_failures.append({"split": split, "sha256": item["sha256"],
                                   "label_id": item["label_id"], "reason": type(exc).__name__})
            continue
        examples.append({**{k: v for k, v in item.items() if k != "html"}, **parsed})
    return examples


train_data = build_examples(train_raw, "train")
val_data = build_examples(val_raw, "val")
test_data = build_examples(test_raw, "test")
for name, data in (("train", train_data), ("val", val_data), ("test", test_data)):
    print(name, len(data), dict(Counter(r["label_id"] for r in data)))
print("parse failures:", dict(Counter(r["split"] for r in parse_failures)))

## 4. Dataset and dataloaders

One page, one label. Tokenisation happens in `__getitem__` so 14,000 padded 2,048-token
tensors never sit in RAM at once; two workers keep the GPU fed.

In [ ]:
processor = MarkupLMProcessor.from_pretrained(BASE_MODEL)
processor.parse_html = False
# Persisted to tokenizer_config.json; left at 512 it would tell loaders the model is narrower.
processor.tokenizer.model_max_length = MAX_TOKENS


def encode_page(item, page_processor, max_length=MAX_TOKENS):
    return page_processor(nodes=item["nodes"], xpaths=item["xpaths"], truncation=True,
                          padding="max_length", max_length=max_length, return_tensors="pt")


class HtmlPhishingDataset(Dataset):
    """One HTML page, one integer label."""
    def __init__(self, data, processor, max_length=MAX_TOKENS):
        self.data, self.processor, self.max_length = data, processor, max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        encoding = {k: v.squeeze(0) for k, v in encode_page(item, self.processor, self.max_length).items()}
        encoding["labels"] = torch.tensor(item["label_id"], dtype=torch.long)
        return encoding


train_dataset = HtmlPhishingDataset(train_data, processor)
val_dataset = HtmlPhishingDataset(val_data, processor)
test_dataset = HtmlPhishingDataset(test_data, processor)

loader_args = {"batch_size": BATCH_SIZE, "num_workers": 2, "pin_memory": True}
train_dataloader = DataLoader(train_dataset, shuffle=True,
                              generator=torch.Generator().manual_seed(SEED), **loader_args)
val_dataloader = DataLoader(val_dataset, **loader_args)
test_dataloader = DataLoader(test_dataset, **loader_args)

token_counts = [int(train_dataset[i]["attention_mask"].sum()) for i in range(min(200, len(train_dataset)))]
print(f"sampled {len(token_counts)} training pages: median {sorted(token_counts)[len(token_counts) // 2]} tokens, "
      f"{sum(c >= MAX_TOKENS for c in token_counts)} hit the {MAX_TOKENS}-token cap")

## 5. Model with a wider context window

The base checkpoint reads 512 tokens. `extend_position_embeddings` tiles its learned
position table out to `MAX_TOKENS` — the same widening MarkupLM `v1` used. Nothing else in
the architecture changes; fine-tuning is what makes the extra positions meaningful.

In [ ]:
def extend_position_embeddings(model, max_tokens):
    """Widen MarkupLM's learned position table by tiling it (Longformer's trick for RoBERTa)."""
    target = max_tokens + 2  # keep RoBERTa's padding_idx offset
    embeddings = model.markuplm.embeddings
    old = embeddings.position_embeddings
    if old.num_embeddings >= target:
        return model
    widened = nn.Embedding(target, old.embedding_dim, padding_idx=old.padding_idx)
    with torch.no_grad():
        widened.weight[:2] = old.weight[:2]  # pad and offset slots are not positions
        learned = old.weight[2:]
        cursor = 2
        while cursor < target:
            span = min(learned.size(0), target - cursor)
            widened.weight[cursor:cursor + span] = learned[:span]
            cursor += span
    embeddings.position_embeddings = widened.to(old.weight.device, old.weight.dtype)
    # position_ids is non-persistent and rebuilt from the config on load.
    embeddings.register_buffer(
        "position_ids", torch.arange(target, device=old.weight.device).expand((1, -1)), persistent=False)
    model.config.max_position_embeddings = target
    return model


def require_finite(tensor, context):
    if not torch.isfinite(tensor).all().item():
        raise RuntimeError(f"Non-finite {context}. Restart training from the base checkpoint.")


def assert_finite_parameters(model):
    for name, parameter in model.named_parameters():
        require_finite(parameter, f"parameter {name}")


def create_model():
    torch.manual_seed(SEED)
    model = MarkupLMForSequenceClassification.from_pretrained(
        BASE_MODEL, num_labels=2, id2label=id2label, label2id=label2id,
        use_safetensors=False, dtype=torch.float32)
    before = model.markuplm.embeddings.position_embeddings.num_embeddings
    extend_position_embeddings(model, MAX_TOKENS)
    after = model.markuplm.embeddings.position_embeddings.num_embeddings
    print(f"position table {before} -> {after}")
    model = model.to(device)
    assert_finite_parameters(model)
    return model

## 6. Metrics

Every evaluation returns phishing probabilities, so AUC is reported alongside the
thresholded numbers. `summarize_scores` matches Laya's, so the two metadata files line up.

In [ ]:
@torch.no_grad()
def evaluate_model(model, dataloader):
    model.eval()
    scores, labels, total_loss = [], [], 0.0
    for step, batch in enumerate(dataloader, start=1):
        inputs = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
        with amp():
            outputs = model(**inputs)
        logits = outputs.logits.float()
        require_finite(logits, f"evaluation logits at batch {step}")
        total_loss += nn.functional.cross_entropy(logits, inputs["labels"], reduction="sum").item()
        scores.extend(torch.softmax(logits, -1)[:, 1].cpu().tolist())
        labels.extend(batch["labels"].tolist())
    if not labels:
        raise ValueError("Cannot evaluate an empty dataset.")
    return total_loss / len(labels), scores, labels


def summarize_scores(labels, scores, threshold):
    predictions = [int(s >= threshold) for s in scores]
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="binary", pos_label=1, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()
    return {"accuracy": accuracy_score(labels, predictions),
            "precision_phishing": precision, "recall_phishing": recall, "f1_phishing": f1,
            "auc": roc_auc_score(labels, scores) if len(set(labels)) == 2 else None,
            "fpr": float(fp / (tn + fp)) if tn + fp else None,
            "confusion": [[int(tn), int(fp)], [int(fn), int(tp)]],
            "threshold": threshold, "n": len(labels)}

## 7. Train and restore the best validation-loss checkpoint

`NUM_EPOCHS` is a maximum. Training stops after two epochs without a lower validation loss,
then the best weights are restored. Each improving epoch is also saved to
`ARTIFACT_DIR/best-checkpoint`, so an interrupted Kaggle session still leaves a usable model.

In [ ]:
for name, data in (("train", train_data), ("val", val_data), ("test", test_data)):
    if set(r["label_id"] for r in data) != {0, 1}:
        raise ValueError(f"{name} must contain both classes")

model = create_model()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
steps_per_epoch = math.ceil(len(train_dataloader) / GRAD_ACCUM_STEPS)
total_steps = steps_per_epoch * NUM_EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=int(total_steps * WARMUP_RATIO), num_training_steps=total_steps)
scaler = torch.amp.GradScaler("cuda", enabled=(AMP_DTYPE == torch.float16))
print(f"{len(train_dataloader)} micro-batches/epoch -> {steps_per_epoch} optimizer steps "
      f"(effective batch {BATCH_SIZE * GRAD_ACCUM_STEPS})")

history, best_state, best_loss, best_epoch, stale = [], None, float("inf"), None, 0
for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    running_loss, correct, seen = 0.0, 0, 0
    started = time.time()
    optimizer.zero_grad(set_to_none=True)
    progress = tqdm(train_dataloader, desc=f"epoch {epoch}/{NUM_EPOCHS}")
    for step, batch in enumerate(progress, start=1):
        inputs = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
        with amp():
            outputs = model(**inputs)
        loss = outputs.loss.float()
        require_finite(loss, f"training loss (epoch {epoch}, batch {step})")
        # Scale so accumulated gradients average rather than sum.
        scaler.scale(loss / GRAD_ACCUM_STEPS).backward()
        if step % GRAD_ACCUM_STEPS == 0 or step == len(train_dataloader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            # float16 overflow makes the scaler skip a step; don't advance the schedule then.
            old_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            if scaler.get_scale() >= old_scale:
                scheduler.step()
            optimizer.zero_grad(set_to_none=True)
        labels = inputs["labels"]
        seen += labels.numel()
        running_loss += loss.item() * labels.numel()
        correct += (outputs.logits.detach().argmax(-1) == labels).sum().item()
        progress.set_postfix(loss=running_loss / seen, accuracy=correct / seen)

    assert_finite_parameters(model)
    val_loss, val_scores, val_labels = evaluate_model(model, val_dataloader)
    entry = {"epoch": epoch, "train_loss": running_loss / seen, "train_accuracy": correct / seen,
             "val": {"loss": val_loss, **summarize_scores(val_labels, val_scores, 0.5)},
             "seconds": round(time.time() - started)}
    history.append(entry)
    print(json.dumps(entry))
    if val_loss < best_loss:
        best_loss, best_epoch, stale = val_loss, epoch, 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        model.save_pretrained(ARTIFACT_DIR / "best-checkpoint", safe_serialization=True)
        processor.save_pretrained(ARTIFACT_DIR / "best-checkpoint")
        print(f"  best so far (epoch {epoch}, val_loss {best_loss:.4f}); saved")
    else:
        stale += 1
        if stale >= EARLY_STOPPING_PATIENCE:
            print("Early stopping: validation loss stopped improving.")
            break

if best_state is None:
    raise RuntimeError("No valid checkpoint was produced")
model.load_state_dict(best_state)
del best_state
model.eval()
print("restored epoch", best_epoch)

## 8. Pick the warn and block cutoffs on validation

The restored model scores every validation page; each score is kept with the page's true
label. For every candidate cutoff (a page is flagged when its score is **at or above** it):

- **phishing caught** = phishing pages flagged ÷ phishing pages
- **safe flagged** = safe pages flagged ÷ safe pages

Then:

- **Block cutoff**: among cutoffs flagging at most `BLOCK_MAX_FPR` of safe pages, the one
  that catches the most phishing.
- **Warn cutoff**: among cutoffs catching at least `WARN_MIN_RECALL` of phishing, the one
  that flags the fewest safe pages.

Only validation is used here. The test set is scored once in section 9 with these cutoffs
fixed, so its numbers stay honest. The validation set has about 1,000 safe pages, so one
wrongly flagged page moves the safe-flagged rate by about 0.1 percentage point.

The scores are rescored here rather than reused from the training loop, because the loop's
last validation pass belongs to the final epoch, not necessarily the restored one.

In [ ]:
_, val_scores, val_labels = evaluate_model(model, val_dataloader)
phishing_scores = [s for s, y in zip(val_scores, val_labels) if y == 1]
safe_scores = [s for s, y in zip(val_scores, val_labels) if y == 0]


def rates_at(cut):
    caught = sum(s >= cut for s in phishing_scores)
    flagged = sum(s >= cut for s in safe_scores)
    return {"cut": cut, "caught": caught / len(phishing_scores), "safe_flagged": flagged / len(safe_scores),
            "caught_n": caught, "safe_flagged_n": flagged}


# Every score is a candidate: moving a cutoff between two scores changes nothing.
# The extra cut above the top score flags nothing, so a block cutoff always exists.
candidates = sorted(set(val_scores)) + [math.nextafter(max(val_scores), math.inf)]
sweep = [rates_at(c) for c in candidates]

block = max((r for r in sweep if r["safe_flagged"] <= BLOCK_MAX_FPR),
            key=lambda r: (r["caught"], -r["safe_flagged"], r["cut"]))
warn = max((r for r in sweep if r["caught"] >= WARN_MIN_RECALL),
           key=lambda r: (-r["safe_flagged"], r["caught"], r["cut"]))
if warn["cut"] > block["cut"]:
    print("The block cutoff already catches the warn target; the warn band is empty.")
    warn = block
BANDS = {"warn_from": warn["cut"], "block_from": block["cut"]}

print(f"validation: {len(phishing_scores)} phishing, {len(safe_scores)} safe\n")
print(f"{'cutoff':>8}  {'phishing caught':>16}  {'safe flagged':>13}")
for cut in (0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.2):
    r = rates_at(cut)
    print(f"{cut:>8.2f}  {r['caught']:>16.1%}  {r['safe_flagged']:>13.1%}")

print("\n               cutoff   phishing caught   safe flagged")
for name, cuts in (("current", CURRENT_BANDS), ("tuned", BANDS)):
    for band in ("warn_from", "block_from"):
        r = rates_at(cuts[band])
        print(f"{name:>7} {band[:-5]:>5}   {r['cut']:.4f}   {r['caught']:>15.1%}   {r['safe_flagged']:>12.1%}")

threshold_selection = {"block_max_fpr": BLOCK_MAX_FPR, "warn_min_recall": WARN_MIN_RECALL,
                       "block": block, "warn": warn,
                       "validation_phishing_n": len(phishing_scores),
                       "validation_safe_n": len(safe_scores)}

## 9. Held-out test and the Laya comparison

Scored once, after the checkpoint and cutoffs are fixed. Reported at the tuned cutoffs, at
today's 0.4 / 0.6, and at 0.5, plus a page-length-only baseline: a model that can't beat
"short pages look suspicious" hasn't learned anything useful. Don't go back and change the
cutoffs because of what this cell shows.

If `LAYA_ARTIFACT_DIR` is set, the test pages are compared by content hash with Laya's
`split_manifest.jsonl`. The AUCs are only put side by side when the test sets match.

In [ ]:
test_loss, test_scores, test_labels = evaluate_model(model, test_dataloader)
test_metrics = summarize_scores(test_labels, test_scores, BANDS["block_from"])
test_metrics_at_half = summarize_scores(test_labels, test_scores, 0.5)
test_bands = {f"{name}_{band}": summarize_scores(test_labels, test_scores, cuts[band])
              for name, cuts in (("tuned", BANDS), ("current", CURRENT_BANDS))
              for band in ("warn_from", "block_from")}
length_baseline = roc_auc_score(test_labels, [-r["html_len"] for r in test_data])

print("                         cutoff   phishing caught   safe flagged")
for key, m in test_bands.items():
    print(f"{key:>22}   {m['threshold']:.4f}   {m['recall_phishing']:>15.1%}   {m['fpr']:>12.1%}")
print("\nAt the tuned block cutoff:", json.dumps(test_metrics, indent=2))
print(f"\nMarkupLM 5k test AUC   {test_metrics['auc']:.3f}")
print(f"page length alone      {length_baseline:.3f}")

slice_metrics = {}
for name, indices in {
    "short_benign": [i for i, r in enumerate(test_data) if r["short"] and r["label_id"] == 0],
    "benign_login": [i for i, r in enumerate(test_data) if r["is_login"] and r["label_id"] == 0],
    "long_pages": [i for i, r in enumerate(test_data) if not r["short"]],
}.items():
    slice_metrics[name] = (summarize_scores([test_labels[i] for i in indices],
                                            [test_scores[i] for i in indices], 0.5)
                           if indices else {"n": 0})
print("Diagnostic slices:", json.dumps(slice_metrics, indent=2))

laya_comparison = None
if LAYA_ARTIFACT_DIR is not None:
    laya_dir = Path(LAYA_ARTIFACT_DIR)
    laya_meta = json.loads((laya_dir / "metadata.json").read_text(encoding="utf-8"))
    with (laya_dir / "split_manifest.jsonl").open(encoding="utf-8") as handle:
        laya_test = {row["sha256"] for row in map(json.loads, handle) if row["split"] == "test"}
    ours = {r["sha256"] for r in test_data}
    laya_auc = (laya_meta.get("test_metrics") or {}).get("auc")
    laya_comparison = {"laya_test_n": len(laya_test), "markuplm_test_n": len(ours),
                       "shared": len(ours & laya_test), "identical": ours == laya_test,
                       "laya_auc": laya_auc, "markuplm_auc": test_metrics["auc"]}
    print("\nTest-set check:", json.dumps(laya_comparison))
    if laya_auc is None:
        print("Laya metadata has no test AUC; compare on Mendeley with smoke_test.ipynb instead.")
    elif ours == laya_test:
        print(f"Same {len(ours)} test pages:  Laya {laya_auc:.3f}  vs  MarkupLM {test_metrics['auc']:.3f}")
    else:
        print("Test pages differ (see counts above), so these AUCs are not a like-for-like comparison.")

## 10. Save the artifact

The layout is the one `backend/app/page_model.py` and `smoke_test.ipynb` already load:
weights, processor, and a `metadata.json` whose `preprocessing` block carries the cleaning
rules and the 2,048-token window. `inference` stores the tuned cutoffs: `operating_threshold`
is the block cutoff (the backend reads it as the phishing verdict) and `risk_bands` holds both.
Scores are not temperature-calibrated, so a score of 0.7 is a ranking, not a 70% probability.

In [ ]:
assert_finite_parameters(model)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(ARTIFACT_DIR, safe_serialization=True)
processor.save_pretrained(ARTIFACT_DIR)

metadata = {
    "base_model": BASE_MODEL,
    "task": "sequence-classification",
    "id2label": id2label,
    "train_dataset": "phreshphish/phreshphish",
    "seed": SEED,
    "learning_rate": LEARNING_RATE, "warmup_ratio": WARMUP_RATIO, "max_grad_norm": MAX_GRAD_NORM,
    "batch_size": BATCH_SIZE, "grad_accum_steps": GRAD_ACCUM_STEPS,
    "effective_batch_size": BATCH_SIZE * GRAD_ACCUM_STEPS,
    "max_epochs": NUM_EPOCHS, "best_epoch": best_epoch,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "selection_metric": "lowest validation cross-entropy",
    "history": history,
    "train_n": len(train_data), "val_n": len(val_data), "test_n": len(test_data),
    "test_metrics": test_metrics,
    "test_metrics_at_half": test_metrics_at_half,
    "test_metrics_by_band": test_bands,
    "threshold_selection": threshold_selection,
    "slice_metrics": slice_metrics,
    "baselines": {"length_only_auc_test": length_baseline},
    "laya_comparison": laya_comparison,
    "preprocessing": {"parse_char_budget": PARSE_CHAR_BUDGET, "strip_tags": STRIP_TAGS,
                      "strip_comments": True, "max_tokens": MAX_TOKENS},
    "context_window": {"max_tokens": MAX_TOKENS, "base_max_tokens": 512,
                       "position_embeddings": model.config.max_position_embeddings,
                       "method": "tiled the base checkpoint's 512 learned position embeddings"},
    # operating_threshold is what backend/app/page_model.py reads as the phishing verdict cut.
    "inference": {"temperature": 1.0, "operating_threshold": BANDS["block_from"],
                  "risk_bands": BANDS, "calibrated": False, "threshold_selected": True},
    "sampling_config": sample_config, "sampling_report": sampling_report,
    "note": "Same sampling as finetune_laya_phishing_kaggle_v2.ipynb, for a like-for-like comparison.",
    "versions": {p: version(p) for p in ("torch", "transformers")},
}
(ARTIFACT_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2, default=str), encoding="utf-8")
with (ARTIFACT_DIR / "split_manifest.jsonl").open("w", encoding="utf-8") as handle:
    for split, data in (("train", train_data), ("val", val_data), ("test", test_data)):
        for r in data:
            handle.write(json.dumps({k: r[k] for k in ("sha256", "host", "shard", "label_id", "short", "is_login")}
                                    | {"split": split}) + "\n")
(ARTIFACT_DIR / "parse_failures.json").write_text(json.dumps(parse_failures, indent=2), encoding="utf-8")
print("saved to", ARTIFACT_DIR.resolve())

## 11. Reload parity check

Reload the saved artifact from disk and compare its logits with the in-memory model on a
real test batch, in float32. A mismatch is a save/load bug, not a model-quality problem.

In [ ]:
loaded_model = MarkupLMForSequenceClassification.from_pretrained(
    ARTIFACT_DIR, use_safetensors=True, dtype=torch.float32).to(device).eval()
parity_batch = next(iter(test_dataloader))
parity_inputs = {k: v.to(device) for k, v in parity_batch.items() if k != "labels"}
with torch.no_grad():
    before = model.float()(**parity_inputs).logits
    after = loaded_model(**parity_inputs).logits
torch.testing.assert_close(before, after, rtol=1e-4, atol=1e-5)
print("PASS: original and reloaded checkpoint agree on a test batch.")

## Next: the fair comparison on Mendeley

1. Download `markuplm-phishing-5k` from the Kaggle Output tab (the `best-checkpoint`
   subfolder can be deleted; it's only a crash backup).
2. Place it at `ai/markuplm/artifacts/markuplm-phishing-5k/`.
3. In `smoke_test.ipynb`, set `MODEL_DIR = Path("artifacts/markuplm-phishing-5k")` and run
   it. Its Mendeley section scores the same 300 pages Laya v3 was scored on.

| Model (same 300 Mendeley pages) | AUC |
|---|---|
| Laya before fine-tuning | 0.724 |
| Laya v3 (fine-tuned) | 0.798 |
| MarkupLM v1 (800 pages) | 0.814 |
| MarkupLM 5k (this notebook) | *fill in* |